In [ ]:
# Top termos TF-IDF por período
import numpy as np
from scipy.sparse import load_npz
import pandas as pd
import json

# Carregar matriz e vocabulário
X = load_npz(os.path.join(PROC_PATH, "tfidf_daily_matrix.npz"))
with open(os.path.join(PROC_PATH, "tfidf_daily_vocab.json"), "r", encoding="utf-8") as f:
    vocab_data = json.load(f)
vocab = np.array(vocab_data['terms'])

# Carregar índice
index_df = pd.read_csv(os.path.join(PROC_PATH, "tfidf_daily_index.csv"))
index_df['date'] = pd.to_datetime(index_df['date'])

# Função para pegar top termos de um período
def get_top_terms_period(start_idx, end_idx, topn=20):
    """Retorna top termos por TF-IDF médio em um período"""
    # Somar TF-IDF de todas as linhas do período
    period_tfidf = X[start_idx:end_idx].mean(axis=0).A1  # .A1 converte matriz 1xN para array
    
    # Top termos
    top_idx = np.argsort(-period_tfidf)[:topn]
    top_terms = [(vocab[i], period_tfidf[i]) for i in top_idx if period_tfidf[i] > 0]
    
    return top_terms

# Análise por ano
print("📊 Top 15 termos TF-IDF por ano:\n")

years = index_df['date'].dt.year.unique()
years = sorted([y for y in years if y >= 2018])

results = []
for year in years:
    # Filtrar índices do ano
    year_mask = index_df['date'].dt.year == year
    year_indices = index_df[year_mask].index.tolist()
    
    if len(year_indices) == 0:
        continue
    
    start_idx = min(year_indices)
    end_idx = max(year_indices) + 1
    
    top_terms = get_top_terms_period(start_idx, end_idx, topn=15)
    
    print(f"{'='*60}")
    print(f"📅 Ano {year} ({len(year_indices)} dias)")
    print(f"{'='*60}")
    
    for i, (term, score) in enumerate(top_terms, 1):
        print(f"{i:2d}. {term:30s} (TF-IDF: {score:.4f})")
        results.append({
            'year': year,
            'rank': i,
            'term': term,
            'tfidf_score': score
        })
    print()

# Salvar resultados
results_df = pd.DataFrame(results)
results_file = os.path.join(PROC_PATH, "top_terms_by_year.csv")
results_df.to_csv(results_file, index=False, encoding='utf-8')

print(f"✅ Resultados salvos em: {results_file}")
print(f"\n⏭️ Próximo passo: Executar Notebook 16 (Models TF-IDF Baselines)")

In [ ]:
# Análise de correlação: Sentiment x Retornos Ibovespa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# Carregar dataset
dataset = pd.read_parquet(os.path.join(PROC_PATH, "dataset_daily_complete.parquet"))
dataset['date'] = pd.to_datetime(dataset['date'])

# Criar figura com 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Série temporal: Sentiment vs Retornos
if 'sentiment' in dataset.columns and 'returns' in dataset.columns:
    ax1 = axes[0]
    ax2 = ax1.twinx()
    
    # Filtrar dados válidos
    valid = dataset[dataset['sentiment'].notna() & dataset['returns'].notna()].copy()
    
    # Rolling means para suavizar
    valid['sentiment_smooth'] = valid['sentiment'].rolling(7, min_periods=1).mean()
    valid['returns_smooth'] = valid['returns'].rolling(7, min_periods=1).mean()
    
    ax1.plot(valid['date'], valid['sentiment_smooth'], color='blue', alpha=0.7, label='Sentiment (7d MA)')
    ax2.plot(valid['date'], valid['returns_smooth']*100, color='green', alpha=0.7, label='Retornos % (7d MA)')
    
    ax1.set_xlabel('Data')
    ax1.set_ylabel('Sentiment', color='blue')
    ax2.set_ylabel('Retornos Ibovespa (%)', color='green')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax2.tick_params(axis='y', labelcolor='green')
    ax1.set_title('Sentiment vs Retornos do Ibovespa (Médias Móveis 7 dias)')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')

# 2. Scatter plot: Sentiment vs Retornos D+1
if 'sentiment' in dataset.columns and 'return_+1d' in dataset.columns:
    valid = dataset[dataset['sentiment'].notna() & dataset['return_+1d'].notna()].copy()
    
    if len(valid) > 0:
        x = valid['sentiment']
        y = valid['return_+1d'] * 100
        
        axes[1].scatter(x, y, alpha=0.3, s=20, c='blue')
        
        # Linha de regressão
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        axes[1].plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2, label=f'y={z[0]:.2f}x+{z[1]:.2f}')
        
        # Correlação
        corr, pval = pearsonr(x, y)
        axes[1].set_title(f'Sentiment vs Retorno D+1 (Correlação: {corr:.3f}, p-value: {pval:.4f})')
        axes[1].set_xlabel('Sentiment')
        axes[1].set_ylabel('Retorno D+1 (%)')
        axes[1].grid(True, alpha=0.3)
        axes[1].axhline(0, color='black', linestyle='-', linewidth=0.5)
        axes[1].axvline(0, color='black', linestyle='-', linewidth=0.5)
        axes[1].legend()

# 3. Distribuição de retornos por quartil de sentiment
if 'sentiment' in dataset.columns and 'return_+1d' in dataset.columns:
    valid = dataset[dataset['sentiment'].notna() & dataset['return_+1d'].notna()].copy()
    
    if len(valid) > 0:
        # Criar quartis
        valid['sentiment_quartile'] = pd.qcut(valid['sentiment'], q=4, labels=['Q1 (Baixo)', 'Q2', 'Q3', 'Q4 (Alto)'])
        
        # Boxplot
        data_to_plot = [valid[valid['sentiment_quartile'] == q]['return_+1d'].values * 100 
                        for q in ['Q1 (Baixo)', 'Q2', 'Q3', 'Q4 (Alto)']]
        
        bp = axes[2].boxplot(data_to_plot, labels=['Q1 (Baixo)', 'Q2', 'Q3', 'Q4 (Alto)'], 
                             patch_artist=True, showmeans=True)
        
        # Colorir boxes
        colors = ['red', 'orange', 'lightblue', 'green']
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        
        axes[2].set_title('Distribuição de Retornos D+1 por Quartil de Sentiment')
        axes[2].set_xlabel('Quartil de Sentiment')
        axes[2].set_ylabel('Retorno D+1 (%)')
        axes[2].grid(True, alpha=0.3, axis='y')
        axes[2].axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
        
        # Adicionar médias
        means = [valid[valid['sentiment_quartile'] == q]['return_+1d'].mean() * 100 
                 for q in ['Q1 (Baixo)', 'Q2', 'Q3', 'Q4 (Alto)']]
        for i, mean in enumerate(means, 1):
            axes[2].text(i, mean, f'{mean:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PROC_PATH, 'sentiment_returns_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Gráficos salvos em: sentiment_returns_analysis.png")

## Visualizações e Análises

In [ ]:
# Verificar outputs gerados
import os
import pandas as pd
import json
from scipy.sparse import load_npz

from src.io import paths
DATA_PATHS = paths.get_data_paths()
PROC_PATH = DATA_PATHS["data_processed"]

print("📦 Verificando arquivos gerados...\n")

# 1. Matriz TF-IDF
tfidf_matrix_file = os.path.join(PROC_PATH, "tfidf_daily_matrix.npz")
if os.path.exists(tfidf_matrix_file):
    X = load_npz(tfidf_matrix_file)
    print(f"✅ tfidf_daily_matrix.npz")
    print(f"   Shape: {X.shape} (dias x features)")
    print(f"   Densidade: {X.nnz / (X.shape[0] * X.shape[1]) * 100:.2f}%")
    print(f"   Tamanho: {os.path.getsize(tfidf_matrix_file) / 1024 / 1024:.1f} MB\n")
else:
    print("❌ tfidf_daily_matrix.npz não encontrado\n")

# 2. Vocabulário
tfidf_vocab_file = os.path.join(PROC_PATH, "tfidf_daily_vocab.json")
if os.path.exists(tfidf_vocab_file):
    with open(tfidf_vocab_file, "r", encoding="utf-8") as f:
        vocab_data = json.load(f)
    print(f"✅ tfidf_daily_vocab.json")
    print(f"   N_features: {vocab_data['n_features']}")
    print(f"   Primeiros 10 termos: {vocab_data['terms'][:10]}\n")
else:
    print("❌ tfidf_daily_vocab.json não encontrado\n")

# 3. Labels
labels_file = os.path.join(PROC_PATH, "labels_y_daily.csv")
if os.path.exists(labels_file):
    labels = pd.read_csv(labels_file)
    print(f"✅ labels_y_daily.csv")
    print(f"   Shape: {labels.shape}")
    print(f"   Colunas: {labels.columns.tolist()}")
    print(f"   Período: {labels['date'].min()} → {labels['date'].max()}")
    
    # Estatísticas dos targets
    for col in labels.columns:
        if col.startswith('target_'):
            total = labels[col].notna().sum()
            positive = labels[col].sum()
            print(f"   {col}: {total} labels, {positive/total*100:.1f}% positivos")
    print()
else:
    print("❌ labels_y_daily.csv não encontrado\n")

# 4. Dataset completo
dataset_file = os.path.join(PROC_PATH, "dataset_daily_complete.parquet")
if os.path.exists(dataset_file):
    dataset = pd.read_parquet(dataset_file)
    print(f"✅ dataset_daily_complete.parquet")
    print(f"   Shape: {dataset.shape}")
    print(f"   Tamanho: {os.path.getsize(dataset_file) / 1024:.1f} KB")
    print(f"   Colunas: {dataset.columns.tolist()}\n")
    
    display(dataset.head(10))
else:
    print("❌ dataset_daily_complete.parquet não encontrado\n")

## Análise dos Outputs

In [ ]:
# Executar pipeline TF-IDF
from src.pipeline.tfidf_features_pipeline import run_tfidf_pipeline

# Parâmetros do pipeline
PARAMS = {
    "min_df": 2,              # Mínimo de documentos para termo ser incluído
    "max_df": 0.95,           # Máximo % de documentos (filtrar palavras muito comuns)
    "ngram_range": (1, 2),    # Unigrams + Bigrams
    "max_features": 5000,     # Limitar vocabulário para performance
    "target_horizons": [1, 3, 5],  # D+1, D+3, D+5
    "rolling_windows": [3, 5, 7]   # Janelas móveis de 3, 5 e 7 dias
}

print("⚙️ Executando pipeline TF-IDF...")
print(f"   Parâmetros: {PARAMS}")
print("\n⚠️ Nota: Download do Ibovespa via yfinance pode demorar ~30 segundos")
print("          Geração da matriz TF-IDF pode demorar ~2-5 minutos\n")

# Executar
report = run_tfidf_pipeline(**PARAMS)

In [ ]:
# Setup de paths
import sys
from pathlib import Path

# Adicionar src ao path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

# 15. Features TF-IDF Diário

## Objetivos
1. **Carregar dados preprocessados** do NB 14 (`news_clean.parquet`)
2. **Baixar dados do Ibovespa** via yfinance (período 2018-2025)
3. **Agregar textos por dia** para criar documentos diários
4. **Gerar matriz TF-IDF** com vocabulário controlado
5. **Criar labels target** (D+1, D+3, D+5) baseados em retornos futuros
6. **Gerar features de janelas móveis** (volatilidade, sentiment rolling)
7. **Análise de correlação** entre sentiment e retornos do Ibovespa

## Outputs
- `data_processed/tfidf_daily_matrix.npz` - Matriz TF-IDF esparsa
- `data_processed/tfidf_daily_vocab.json` - Vocabulário e metadados
- `data_processed/tfidf_daily_index.csv` - Índice date → row_id
- `data_processed/labels_y_daily.csv` - Labels target (D+1, D+3, D+5)
- `data_processed/dataset_daily_complete.parquet` - Dataset completo com todas as features
- `data_processed/ibovespa_YYYYMMDD_HHMMSS.csv` - Dados históricos do Ibovespa
- `data_processed/tfidf_report_YYYYMMDD_HHMMSS.json` - Relatório detalhado